# Worksheet 3: Regression & Classification
## Wine Quality Prediction (Red Wine)

**Dataset:** [Red Wine Quality (Cortez et al., 2009) — Kaggle](https://www.kaggle.com/datasets/uciml/red-wine-quality-cortez-et-al-2009)
(Original source: [UCI Machine Learning Repository - Wine Quality](https://archive.ics.uci.edu/dataset/186/wine+quality))

This notebook covers every topic in the lab worksheet:
- Worksheet 2: Data Preprocessing
- LAB 1: Regression (Simple Linear Regression, Multiple Linear Regression, Quality Score Prediction)
- LAB 2: Classification (Preparing Data, Decision Boundary, Logistic Regression, Good/Bad Wine Prediction, Confusion Matrix)
- LAB 3: Model Comparison (Simple vs Multiple, Training vs Testing, Regression vs Classification, Performance Metrics)

The dataset contains 1,599 red wine samples, each described by 11 physicochemical
measurements (fixed acidity, volatile acidity, citric acid, residual sugar, chlorides,
free/total sulfur dioxide, density, pH, sulphates, alcohol) plus a **quality score (0-10)**
given by wine tasters.

- **Regression target:** `quality` (continuous score) — used in LAB 1
- **Classification target:** derived `good_wine` label (1 = quality >= 7, "good"; 0 = otherwise) — used in LAB 2

---
### How to use this notebook (important)
1. Download `winequality-red.csv` from Kaggle (`uciml/red-wine-quality-cortez-et-al-2009`) and place it
   next to this notebook, OR download the semicolon-separated CSV directly from the UCI repository.
2. If using a **Kaggle Notebook**: add the dataset and set `data = "/kaggle/input/red-wine-quality-cortez-et-al-2009/winequality-red.csv"`
3. If the real CSV is not found, this notebook will **automatically generate a synthetic demo dataset
   (DEMO MODE)** with the same columns and realistic value ranges, so every cell can be run and verified
   before you switch to the real dataset (the numeric results in DEMO MODE are only for testing the
   pipeline, not real conclusions about wine quality).


In [ ]:
# ==========================================================
# 0. Import Libraries
# ==========================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc, classification_report
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True


---
## Worksheet 2: Data Preprocessing

Prepare the data for model building by selecting suitable features, and apply
Principal Component Analysis (PCA) to reduce the number of features and improve
learning efficiency.


In [ ]:
# ==========================================================
# 1. Load data / set up DEMO MODE if the real dataset isn't available yet
# ==========================================================
DATA_PATH = "./winequality-red.csv"   # <-- change this to the path of the real dataset

COLUMNS = ["fixed acidity", "volatile acidity", "citric acid", "residual sugar",
           "chlorides", "free sulfur dioxide", "total sulfur dioxide", "density",
           "pH", "sulphates", "alcohol", "quality"]

DEMO_MODE = not os.path.isfile(DATA_PATH)

if DEMO_MODE:
    print("!! Real dataset not found at path:", DATA_PATH)
    print("!! Generating a synthetic demo dataset (DEMO MODE) with realistic value ranges")

    rng = np.random.default_rng(RANDOM_STATE)
    n = 1599  # same size as the real red wine dataset

    alcohol = rng.normal(10.42, 1.07, n).clip(8.0, 15.0)
    volatile_acidity = rng.normal(0.53, 0.18, n).clip(0.05, 1.6)

    df = pd.DataFrame({
        "fixed acidity": rng.normal(8.32, 1.74, n).clip(4.0, 16.0),
        "volatile acidity": volatile_acidity,
        "citric acid": rng.normal(0.27, 0.19, n).clip(0.0, 1.0),
        "residual sugar": rng.normal(2.54, 1.41, n).clip(0.5, 16.0),
        "chlorides": rng.normal(0.087, 0.047, n).clip(0.01, 0.6),
        "free sulfur dioxide": rng.normal(15.87, 10.46, n).clip(1.0, 72.0),
        "total sulfur dioxide": rng.normal(46.47, 32.90, n).clip(6.0, 289.0),
        "density": rng.normal(0.9967, 0.0019, n).clip(0.990, 1.004),
        "pH": rng.normal(3.311, 0.154, n).clip(2.7, 4.0),
        "sulphates": rng.normal(0.658, 0.170, n).clip(0.3, 2.0),
        "alcohol": alcohol,
    })

    # quality weakly depends on alcohol (+) and volatile acidity (-), like the real dataset,
    # plus noise, so the demo models have something to learn
    quality_raw = (5.0 + 0.35 * (alcohol - alcohol.mean()) / alcohol.std()
                   - 0.30 * (volatile_acidity - volatile_acidity.mean()) / volatile_acidity.std()
                   + rng.normal(0, 0.6, n))
    df["quality"] = np.clip(np.round(quality_raw), 3, 8).astype(int)

    demo_path = "./_demo_winequality.csv"
    df.to_csv(demo_path, index=False)
    print("Synthetic demo dataset created:", demo_path, "-", df.shape)
else:
    df = pd.read_csv(DATA_PATH, sep=";")
    print("Real dataset loaded:", DATA_PATH, "-", df.shape)

df.head()


In [ ]:
# ==========================================================
# 2. Create the Classification label: good_wine (1 = quality >= 7, 0 = otherwise)
# ==========================================================
df["good_wine"] = (df["quality"] >= 7).astype(int)

print("Total number of samples:", len(df))
print("\nQuality score distribution:")
print(df["quality"].value_counts().sort_index())
print("\ngood_wine label distribution:")
print(df["good_wine"].value_counts())


In [ ]:
# ==========================================================
# 3. Exploratory Data Analysis (EDA)
# ==========================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["quality"], bins=range(3, 10), color="firebrick", edgecolor="white", align="left")
axes[0].set_title("Wine Quality Score Distribution")
axes[0].set_xlabel("Quality (0-10)")
axes[0].set_ylabel("Count")

good_counts = df["good_wine"].map({0: "Not good (<7)", 1: "Good (>=7)"}).value_counts()
axes[1].bar(good_counts.index, good_counts.values, color=["#4C72B0", "#DD8452"])
axes[1].set_title("Good vs Not-good Wine Distribution")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()


### Feature Selection with Principal Component Analysis (PCA)

The dataset has 11 physicochemical features, several of which are correlated with each
other (e.g. free sulfur dioxide and total sulfur dioxide, or density and alcohol). We use
**PCA** to reduce the number of features while retaining as much of the data's variance
as possible, which reduces redundancy and can improve model stability.


In [ ]:
# ==========================================================
# 4. Split data into Train/Test first (to avoid data leakage), then apply Scaling + PCA
# ==========================================================
feature_cols = ["fixed acidity", "volatile acidity", "citric acid", "residual sugar",
                 "chlorides", "free sulfur dioxide", "total sulfur dioxide", "density",
                 "pH", "sulphates", "alcohol"]

X = df[feature_cols].values
y_quality = df["quality"].values
y_good = df["good_wine"].values

X_train_raw, X_test_raw, y_quality_train, y_quality_test, y_good_train, y_good_test = train_test_split(
    X, y_quality, y_good, test_size=0.2, random_state=RANDOM_STATE, stratify=y_good
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

N_COMPONENTS = 6
pca = PCA(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"Reduced number of features from {X_train_raw.shape[1]} down to {N_COMPONENTS} dimensions")
print(f"Total explained variance ratio: {pca.explained_variance_ratio_.sum():.2%}")

plt.figure()
plt.plot(np.cumsum(pca.explained_variance_ratio_), marker="o")
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("PCA: Cumulative Explained Variance")
plt.show()


---
## LAB 1: Regression

Predict a continuous value — the **wine quality score** — from physicochemical features.


### 1.1 Simple Linear Regression
Use only 1 feature (the 1st Principal Component, which is dominated by alcohol and
acidity-related measurements) to predict the quality score.


In [ ]:
# ==========================================================
# LAB 1.1: Simple Linear Regression (1 feature: PC1 -> quality)
# ==========================================================
simple_lr = LinearRegression()
simple_lr.fit(X_train_pca[:, [0]], y_quality_train)

pred_quality_simple_train = simple_lr.predict(X_train_pca[:, [0]])
pred_quality_simple_test = simple_lr.predict(X_test_pca[:, [0]])

mae_simple = mean_absolute_error(y_quality_test, pred_quality_simple_test)
rmse_simple = mean_squared_error(y_quality_test, pred_quality_simple_test) ** 0.5
r2_simple = r2_score(y_quality_test, pred_quality_simple_test)

print("=== Simple Linear Regression (PC1 -> Quality) ===")
print(f"MAE  : {mae_simple:.3f}")
print(f"RMSE : {rmse_simple:.3f}")
print(f"R^2  : {r2_simple:.3f}")

# scatter plot + regression line
plt.figure()
order = np.argsort(X_test_pca[:, 0])
plt.scatter(X_test_pca[:, 0], y_quality_test, alpha=0.5, label="Actual")
plt.plot(X_test_pca[order, 0], pred_quality_simple_test[order], color="red", linewidth=2, label="Predicted line")
plt.xlabel("PC1")
plt.ylabel("Quality")
plt.title("Simple Linear Regression: PC1 vs Quality")
plt.legend()
plt.show()


### 1.2 Multiple Linear Regression
Use several features (all Principal Components) to predict the quality score.


In [ ]:
# ==========================================================
# LAB 1.2: Multiple Linear Regression (all PCA components -> quality)
# ==========================================================
multi_lr = LinearRegression()
multi_lr.fit(X_train_pca, y_quality_train)

pred_quality_multi_train = multi_lr.predict(X_train_pca)
pred_quality_multi_test = multi_lr.predict(X_test_pca)

mae_multi = mean_absolute_error(y_quality_test, pred_quality_multi_test)
rmse_multi = mean_squared_error(y_quality_test, pred_quality_multi_test) ** 0.5
r2_multi = r2_score(y_quality_test, pred_quality_multi_test)

print("=== Multiple Linear Regression (All PCs -> Quality) ===")
print(f"MAE  : {mae_multi:.3f}")
print(f"RMSE : {rmse_multi:.3f}")
print(f"R^2  : {r2_multi:.3f}")


### 1.3 Quality Score Prediction (Regression model summary)
Compare the actual quality score against the score predicted by the model, on the test set.


In [ ]:
# ==========================================================
# LAB 1.3: Quality Prediction - Actual vs Predicted (Multiple LR model)
# ==========================================================
plt.figure()
plt.scatter(y_quality_test, pred_quality_multi_test, alpha=0.5)
lims = [min(y_quality_test.min(), pred_quality_multi_test.min()),
        max(y_quality_test.max(), pred_quality_multi_test.max())]
plt.plot(lims, lims, color="red", linestyle="--", label="Perfect prediction")
plt.xlabel("Actual Quality")
plt.ylabel("Predicted Quality")
plt.title("Quality Prediction: Actual vs Predicted (Multiple Linear Regression)")
plt.legend()
plt.show()

# a few individual predictions
sample_results = pd.DataFrame({
    "Actual Quality": y_quality_test[:10],
    "Predicted Quality": np.round(pred_quality_multi_test[:10], 2)
})
sample_results


---
## LAB 2: Classification

Classify wines as **good (1, quality >= 7) vs not good (0, quality < 7)**.


### 2.1 Preparing Classification Data
Reuse the PCA feature set and the derived `good_wine` label already prepared during
Data Preprocessing.


In [ ]:
# ==========================================================
# LAB 2.1: Preparing Classification Data
# ==========================================================
print("Training set:", X_train_pca.shape, y_good_train.shape)
print("Testing set :", X_test_pca.shape, y_good_test.shape)

print("\nClass proportion in the training set:")
print(pd.Series(y_good_train).map({0: "Not good", 1: "Good"}).value_counts(normalize=True))


### 2.2 Decision Boundary Visualization
To visualize the decision boundary, we train a Logistic Regression model using only the
first 2 features (PC1, PC2).


In [ ]:
# ==========================================================
# LAB 2.2: Decision Boundary Visualization (2D: PC1, PC2)
# ==========================================================
logreg_2d = LogisticRegression(max_iter=1000, class_weight="balanced")
logreg_2d.fit(X_train_pca[:, :2], y_good_train)

x_min, x_max = X_train_pca[:, 0].min() - 1, X_train_pca[:, 0].max() + 1
y_min, y_max = X_train_pca[:, 1].min() - 1, X_train_pca[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
Z = logreg_2d.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.figure(figsize=(7, 5))
plt.contourf(xx, yy, Z, alpha=0.25, cmap="coolwarm")
scatter = plt.scatter(X_train_pca[:, 0], X_train_pca[:, 1], c=y_good_train,
                       cmap="coolwarm", edgecolor="k", alpha=0.8)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Decision Boundary: Logistic Regression (Good vs Not-good Wine)")
plt.legend(handles=scatter.legend_elements()[0], labels=["Not good (0)", "Good (1)"])
plt.show()


### 2.3 Logistic Regression (Full model)
Train a Logistic Regression model using all PCA features, for maximum accuracy.
Since "good" wines are a minority class, `class_weight="balanced"` is used to reduce bias
toward the majority class.


In [ ]:
# ==========================================================
# LAB 2.3: Logistic Regression (all PCA components)
# ==========================================================
logreg = LogisticRegression(max_iter=1000, class_weight="balanced")
logreg.fit(X_train_pca, y_good_train)

pred_good_train = logreg.predict(X_train_pca)
pred_good_test = logreg.predict(X_test_pca)
proba_good_test = logreg.predict_proba(X_test_pca)[:, 1]

print("Logistic Regression training complete")


### 2.4 Good/Bad Wine Prediction
Evaluate the good-vs-not-good wine classification performance on the test set.


In [ ]:
# ==========================================================
# LAB 2.4: Good/Bad Wine Prediction - Evaluation
# ==========================================================
acc = accuracy_score(y_good_test, pred_good_test)
prec = precision_score(y_good_test, pred_good_test, zero_division=0)
rec = recall_score(y_good_test, pred_good_test, zero_division=0)
f1 = f1_score(y_good_test, pred_good_test, zero_division=0)

print("=== Good/Bad Wine Prediction (Test set) ===")
print(f"Accuracy  : {acc:.3f}")
print(f"Precision : {prec:.3f}")
print(f"Recall    : {rec:.3f}")
print(f"F1-score  : {f1:.3f}")
print()
print(classification_report(y_good_test, pred_good_test, target_names=["Not good", "Good"], zero_division=0))


### 2.5 Confusion Matrix

In [ ]:
# ==========================================================
# LAB 2.5: Confusion Matrix
# ==========================================================
cm = confusion_matrix(y_good_test, pred_good_test)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Not good", "Good"])
fig, ax = plt.subplots(figsize=(5, 5))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
plt.title("Confusion Matrix: Good/Bad Wine Prediction")
plt.show()


---
## LAB 3: Model Comparison

Compare and discuss model results using appropriate metrics.


### 3.1 Simple vs Multiple Linear Regression

In [ ]:
# ==========================================================
# LAB 3.1: Simple vs Multiple Linear Regression
# ==========================================================
comparison_reg = pd.DataFrame({
    "Model": ["Simple Linear Regression (1 feature)", "Multiple Linear Regression (all PCs)"],
    "MAE": [mae_simple, mae_multi],
    "RMSE": [rmse_simple, rmse_multi],
    "R2": [r2_simple, r2_multi]
})
comparison_reg


> **Discussion:** Multiple Linear Regression uses several dimensions, so it can usually
> capture more complex relationships than Simple Linear Regression, which relies on only
> 1 feature. This typically results in lower error (MAE/RMSE) and a higher R² score.
> Wine quality is a subjective sensory score influenced by many interacting chemical
> factors, so a single feature (PC1) rarely explains it well — R² is often low even for
> the multiple-feature model, which is a realistic and expected finding for this dataset.


### 3.2 Training vs Testing Performance

In [ ]:
# ==========================================================
# LAB 3.2: Training vs Testing Performance
# ==========================================================
def reg_metrics(y_true, y_pred):
    return (mean_absolute_error(y_true, y_pred),
            mean_squared_error(y_true, y_pred) ** 0.5,
            r2_score(y_true, y_pred))

mae_tr, rmse_tr, r2_tr = reg_metrics(y_quality_train, pred_quality_multi_train)
mae_te, rmse_te, r2_te = reg_metrics(y_quality_test, pred_quality_multi_test)

train_test_reg = pd.DataFrame({
    "Set": ["Training", "Testing"],
    "MAE": [mae_tr, mae_te],
    "RMSE": [rmse_tr, rmse_te],
    "R2": [r2_tr, r2_te]
})
print("Regression (Quality) - Multiple Linear Regression")
display(train_test_reg)

acc_tr = accuracy_score(y_good_train, pred_good_train)
acc_te = accuracy_score(y_good_test, pred_good_test)
train_test_clf = pd.DataFrame({
    "Set": ["Training", "Testing"],
    "Accuracy": [acc_tr, acc_te]
})
print("\nClassification (Good/Bad Wine) - Logistic Regression")
display(train_test_clf)


> **Discussion:** If the metrics on the Training set are much better than on the Testing
> set (e.g. a large gap in Accuracy or R²), this can be a sign of **overfitting** — the model
> has memorized the training data too closely and fails to generalize well to new data.
> Consider reducing model complexity, adding more data, or applying regularization.


### 3.3 Regression vs Classification

In [ ]:
# ==========================================================
# LAB 3.3: Regression vs Classification (conceptual comparison)
# ==========================================================
concept_comparison = pd.DataFrame({
    "Aspect": [
        "Output type",
        "Example task in this notebook",
        "Label type",
        "Main model function",
        "Main evaluation metrics",
        "Example training loss/error"
    ],
    "Regression": [
        "Continuous value",
        "Quality Score Prediction (0-10)",
        "Real number",
        "Linear Regression",
        "MAE, RMSE, R^2",
        "Mean Squared Error (MSE)"
    ],
    "Classification": [
        "Category/class",
        "Good vs Not-good Wine Prediction",
        "Discrete class label (0, 1)",
        "Logistic Regression",
        "Accuracy, Precision, Recall, F1, ROC-AUC",
        "Log-loss / Cross-Entropy"
    ]
})
concept_comparison


### 3.4 Model Performance Metrics (Summary + ROC Curve / AUC)

In [ ]:
# ==========================================================
# LAB 3.4: Model Performance Metrics - ROC Curve & AUC
# ==========================================================
fpr, tpr, thresholds = roc_curve(y_good_test, proba_good_test)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC curve (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], color="navy", lw=1, linestyle="--", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve: Good/Bad Wine Classification (Logistic Regression)")
plt.legend(loc="lower right")
plt.show()

print(f"AUC Score: {roc_auc:.3f}")


In [ ]:
# ==========================================================
# LAB 3.4 (continued): Summary table of all Classification metrics
# ==========================================================
metrics_summary = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score", "AUC"],
    "Score": [acc, prec, rec, f1, roc_auc]
})
metrics_summary["Score"] = metrics_summary["Score"].round(3)
metrics_summary


---
## Summary & Building Your Portfolio

- The **Regression** model predicts the wine quality score (a continuous value 0-10),
  evaluated with MAE, RMSE, R²
- The **Classification** model predicts whether a wine is "good" (quality >= 7), evaluated
  with Accuracy, Precision, Recall, F1, ROC/AUC
- **PCA** reduces 11 correlated physicochemical features down to a smaller set of
  components while retaining most of the variance
- Comparing Training vs Testing performance helps check for Overfitting/Underfitting
- Because "good" wines are rare (imbalanced classes), Precision/Recall/F1/AUC are more
  informative than Accuracy alone for the classification task

### Next steps for your GitHub Portfolio
1. Point `data` to the real `winequality-red.csv` and re-run the whole notebook (Restart & Run All)
2. Optionally repeat the analysis on `winequality-white.csv` and compare results
3. Add a `README.md` describing the objective, dataset, how to run it, and key results/plots
4. Push it to a GitHub repository along with screenshots of the key plots and Confusion Matrix
